# Занятие 8, пара 1. Разметка: инструменты и форматы


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Модель учится на наших договорённостях

В 06 мы разделяли изображение на области, в 07 находили и сопровождали объекты.
Но откуда берётся «правильный ответ», с которым сравнивают алгоритм? Его кто-то должен определить.
Сегодня сами пройдём небольшой цикл разметки: **задача → правило → пример → проверка разногласий → файл**.

После пары вы сможете отличать метку изображения от рамки и маски, читать координаты COCO/YOLO,
находить несогласованность разметки и начать инструкцию своего проекта.
Основной путь — NumPy, Matplotlib и стандартный Python. OpenCV, аккаунт и онлайн-разметчик не требуются.
Операции Python знакомы; подробно разбираем именно смысл разметки.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Одна фотография, три разных задания

Откроем знакомую фотографию чашки. Это фото Rachel Michetti (CC0), поставляемое в комплекте;
происхождение записано в `course_support/data/segmentation/SOURCE.md`.
`imshow` показывает массив RGB, а размер фигуры делает его хорошо видимым на проекторе.

Сначала решим, что мы хотим получить:

| Задача | Ответ разметчика | Пример |
|---|---|---|
| Классификация изображения | Метка всей картинки | «Есть чашка» |
| Детекция | Класс и рамка каждого объекта | Чашка здесь, ложка там |
| Сегментация | Метка каждого пикселя/области | Какие пиксели принадлежат чашке |

Рамка отвечает «где примерно», но включает фон. Маска точнее описывает форму, зато требует больше труда.
Универсальной «правильной разметки картинки» нет — есть разметка под конкретную задачу.


In [ ]:
import json
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

photo = np.load(DATA_DIR / 'segmentation' / 'coffee.npy')
height, width = photo.shape[:2]
print('Ширина:', width, 'высота:', height)
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(photo)
ax.set_title('Что считать чашкой: край, корпус, ручку, блюдце?')
plt.show()


### 2. Инструкция до первого прямоугольника

Для учебной детекции договоримся так:

- Класс **cup** — видимая чашка вместе с ручкой. Блюдце и содержимое как отдельные объекты не размечаем.
- Класс **spoon** — ложка: рамка охватывает все её видимые части. Скрытые границы за чашкой не дорисовываем.
- Рамка плотно охватывает видимые части выбранного объекта; внутри неё могут оказаться фон и другие объекты.
- Если класс нельзя надёжно определить, пример идёт на обсуждение, а не получает выдуманную метку.

Последнее правило надо уточнять под будущий проект: что считать допустимой видимостью и как хранить спорные случаи.
Наши рамки ниже нарисованы вручную для обучения, это **не официальный эталон** фотографии.

Координаты: `x` растёт вправо, `y` вниз. Для рамки `(x1, y1, x2, y2)` ширина равна `x2−x1`,
высота `y2−y1`. Считаем координаты границ непрерывными: правую и нижнюю границы не прибавляем как ещё один пиксель.
Это согласуется со срезом `photo[y1:y2, x1:x2]` для целых границ.


Для рисунка используем `Rectangle((x, y), width, height)`: он создаёт прямоугольник по углу и размерам.
`fill=False` оставляет контур; `ax.add_patch` добавляет его поверх фото. Цвет и linewidth задают оформление,
не меняют координаты. `ax.text(x,y,текст)` размещает подпись у указанной точки.


In [ ]:
cup_box = [172, 18, 412, 310]
x1, y1, x2, y2 = cup_box
box_width = x2 - x1
box_height = y2 - y1
print('Размер рамки:', box_width, box_height)
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(photo)
ax.add_patch(Rectangle((x1, y1), box_width, box_height,
                       fill=False, edgecolor='lime', linewidth=3))
ax.set_title('Учебная рамка cup: с ручкой, без блюдца')
plt.show()


`Rectangle((x, y), width, height)` создаёт прямоугольник по левому верхнему углу и размерам;
`fill=False` оставляет только контур. `ax.add_patch` добавляет его поверх изображения.
Этот технический способ рисования не выбирает объект за нас.

Посмотрим два варианта: A охватывает в основном верхнюю часть чашки, B — чашку с ручкой.
Разница кажется геометрической, но сначала нужно проверить договорённость о том, **что входит в объект**.


In [ ]:
box_a = [172, 18, 412, 220]
box_b = [172, 18, 412, 310]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, box, title in zip(axes, [box_a, box_b], ['A', 'B']):
    x1, y1, x2, y2 = box
    ax.imshow(photo)
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                           fill=False, edgecolor='lime', linewidth=3))
    ax.set_title(title)
plt.show()


### ✏️ Ваш ход — Исправить правило, а не усреднить рамки

Разметчик A написал «обвёл чашку», разметчик B — то же самое.
Выберите `chosen_box` по нашей инструкции: `box_a` или `box_b`.
Напишите `annotation_rule` — конкретное правило, которое предотвратит это разногласие на следующем фото.
Отдельно ответьте: стоит ли усреднять координаты A и B, если причиной была разная трактовка объекта?
Проверка проверит выбор и наличие текста, но смысл правила обсуждаем сами.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.1.1', chosen_box, annotation_rule)


**Мой вывод:** …


### 3. Рамку мало нарисовать — её надо связать с изображением

В COCO-подобной структуре есть три таблицы: `images` описывает изображения,
`categories` — классы, `annotations` — объекты. Связи идут по ID, а не по порядку строк.
Одна картинка может иметь много объектов; у каждого объекта свой `id`.

В поле `bbox` COCO записывает **[x, y, width, height]** в пикселях, а не правый нижний угол.
Для чашки из примера это `[172, 18, 240, 292]`.
Ниже минимальная учебная структура для рамок. Полные экспорты могут содержать маски, флаги и дополнительные поля;
здесь они не нужны. Для сегментации COCO поле `area` относится к площади объекта, и её нельзя автоматически
подменять площадью рамки.


In [ ]:
dataset = {
    'images': [{'id': 10, 'file_name': 'coffee.png', 'width': width, 'height': height}],
    'categories': [{'id': 1, 'name': 'cup'}, {'id': 2, 'name': 'spoon'}],
    'annotations': [
        {'id': 100, 'image_id': 10, 'category_id': 1, 'bbox': [172, 18, 240, 292]},
        {'id': 101, 'image_id': 10, 'category_id': 2, 'bbox': [325, 65, 99, 263]},
    ],
}
print(json.dumps(dataset, ensure_ascii=False, indent=2))


Прочитаем одну запись руками. `category_id=2` отсылает к ложке, `image_id=10` — к фотографии.
Последнее число 263 — **высота**, поэтому нижняя граница ложки будет `65+263=328`.
Проверять форматы надо рисунком: корректный JSON вполне может содержать геометрически неверные рамки.


In [ ]:
category_names = {}
for category in dataset['categories']:
    category_names[category['id']] = category['name']

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(photo)
for annotation in dataset['annotations']:
    x, y, w, h = annotation['bbox']
    name = category_names[annotation['category_id']]
    ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor='cyan', linewidth=2))
    ax.text(x, y, name, color='black', backgroundcolor='white')
ax.set_title('Разметка после чтения записей')
plt.show()


До сохранения проверим три вещи: ссылка на изображение существует, класс известен,
размеры положительны и рамка находится внутри изображения. `assert` здесь — проверка согласованности
наших учебных данных; если она не прошла, файл надо исправить, а не отключить проверку.
Сохраняем в outputs, исходные данные не перезаписываем. `plt.imsave` сохранит RGB-массив как PNG без осей; файл coffee.png положим рядом с JSON, чтобы ссылка file_name действительно работала.


In [ ]:
for annotation in dataset['annotations']:
    x, y, w, h = annotation['bbox']
    assert annotation['image_id'] == 10
    assert annotation['category_id'] in category_names
    assert w > 0 and h > 0
    assert 0 <= x < x+w <= width
    assert 0 <= y < y+h <= height

output_dir = WORKSPACE_DIR / 'outputs' / 'annotation_practice'
output_dir.mkdir(parents=True, exist_ok=True)
plt.imsave(output_dir / 'coffee.png', photo)
annotation_path = output_dir / 'boxes_example.json'
annotation_path.write_text(json.dumps(dataset, ensure_ascii=False, indent=2), encoding='utf-8')
print(annotation_path)


### 4. Те же координаты на другом языке

В распространённом формате YOLO строка объекта — `class_id cx cy w h`.
Центр, ширина и высота записаны в долях размеров изображения. Для класса обычно используются
номера от нуля; наш COCO ID 1 нельзя просто скопировать без таблицы соответствия.

Возьмём чашку. Центр по x находится на полпути между левой и правой границей:
`cx = x + w/2`. Затем делим горизонтальные величины на ширину изображения,
вертикальные — на высоту. Деление всего на одно число испортит рамку на **не квадратном** изображении. Фото у нас как раз 600×400.


In [ ]:
x, y, w, h = dataset['annotations'][0]['bbox']
cx_pixels = x + w / 2
cy_pixels = y + h / 2
yolo_box = [cx_pixels / width, cy_pixels / height, w / width, h / height]
yolo_class = 0  # В нашем экспорте 0=cup, 1=spoon.
print('YOLO:', yolo_class, *yolo_box)

cx, cy, nw, nh = yolo_box
restored_w = nw * width
restored_h = nh * height
restored_x = cx * width - restored_w / 2
restored_y = cy * height - restored_h / 2
print('Обратно в COCO:', restored_x, restored_y, restored_w, restored_h)


Проверка обратного преобразования ловит часть ошибок формата. Но если мы дважды перепутали ширину
и высоту одинаковым способом, формальное «туда-обратно» может обмануть. Поэтому нужен ещё рисунок.

Справочник: [YOLO detection format](https://docs.ultralytics.com/datasets/detect/),
[COCO data format](https://cocodataset.org/#format-data).
Pascal VOC — XML с углами `xmin,ymin,xmax,ymax`; при конвертации дополнительно проверяют
соглашение конкретного экспортёра о начале отсчёта и включённой границе. Заучивать все форматы не требуется.


### ✏️ Ваш ход — Найти ошибку после уменьшения изображения

Изображение 600×400 уменьшили вдвое без обрезки. Рамка COCO была `[60, 40, 180, 120]`.
Коллега оставил её пиксельные координаты без изменения: «объект ведь тот же».

Получите `resized_box` в новом изображении и `relative_box` — исходную рамку YOLO без class_id.
Объясните, почему относительные координаты при **таком** преобразовании сохраняются,
а после обрезки изображения обычно нет. Не пользуйтесь `yolo_box` чашки: это другой объект.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.1.2', resized_box, relative_box)


**Мой вывод:** …


### 5. А в своём проекте обязательно рисовать рамки?

Нет. Основной проект курса — **классификация изображений**. Для него часто достаточно CSV:
путь к изображению, класс и группа происхождения. Рамки сегодня нужны, чтобы понять разметку
и следующую пару про метрики детекции. Они не становятся скрытым дополнительным требованием ЛР4.

В наборе обложек из 06 одна и та же обложка встречается в нескольких вариантах яркости.
Её `object_id` — группа, варианты — разные изображения. Чтобы не спутать файлы с независимыми объектами,
сохраним оба идентификатора. `ImageFolder` — альтернативный вариант, где класс задаёт имя папки;
группы всё равно нужно хранить отдельно.


In [ ]:
covers = np.load(DATA_DIR / 'segmentation' / 'covers.npz')
images = covers['images']
labels = covers['labels']
object_ids = covers['object_ids']
print('Изображений:', len(images), 'разных объектов:', len(np.unique(object_ids)))
print('Первые группы:', object_ids[:8])


Экспортируем небольшой пример **синтетических** изображений в PNG и каталог CSV.
`plt.imsave` сохраняет массив как изображение без осей и заголовков; подписи не должны случайно
стать входными признаками будущей модели. `csv.DictWriter` записывает словари с указанными полями,
`writeheader()` — названия столбцов. Относительный путь означает путь от папки каталога.

Это пример устройства файлов, не готовый датасет вашего проекта. Для реального набора нужны
собственные правила классов, разрешённые источники и проверка покрытия.


In [ ]:
image_dir = output_dir / 'images'
image_dir.mkdir(exist_ok=True)
rows = []
for index in range(len(images)):
    filename = f'cover_{index:02}.png'
    plt.imsave(image_dir / filename, images[index])
    rows.append({
        'path': 'images/' + filename,
        'label': int(labels[index]),
        'group_id': str(object_ids[index]),
    })

with (output_dir / 'labels.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=['path', 'label', 'group_id'])
    writer.writeheader()
    writer.writerows(rows)
print(rows[:4])


### 6. Два вида проверки, которые нельзя подменять друг другом

**Согласованность разметки:** двум людям дают одни изображения и одну инструкцию; затем разбирают
разногласия. Высокое совпадение не гарантирует истину, если оба одинаково поняли плохую инструкцию.
Для пилота достаточно небольшой разнообразной выборки, где есть и простые, и спорные случаи.

**Независимость оценки модели:** изображения одного объекта или одной серии не должны оказаться
одновременно в обучении и проверке. Иначе проверяем узнавание почти знакомого кадра.
В 09 разберём разбиение подробно; сейчас сохраним group_id, чтобы такая проверка вообще была возможна.
Множества удобны для проверки: `set(...)` оставляет уникальные значения, `A & B` находит общие.


In [ ]:
train_groups = {10, 11, 12}
validation_groups = {12, 13}
print('Общие объекты:', train_groups & validation_groups)


### ✏️ Ваш ход — Файлы разные, утечка та же

План разбиения:

| Файл | Объект/серия | Часть |
|---|---|---|
| cup_front.png | cup_1 | train |
| cup_side.png | cup_1 | validation |
| spoon_1.png | spoon_1 | train |
| cup_new.png | cup_2 | test |

Напишите `leaked_groups` как множество групп, нарушающих независимость, и `repair_plan` —
как исправить разбиение. Не предлагайте просто переименовать файлы. Что делать, если после исправления
для одного класса остаётся всего один независимый объект?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.1.3', leaked_groups, repair_plan)


**Мой вывод:** …


## ЛР4 — пилот датасета проекта

Откройте `docs/ЛР4.md` в корне своей копии: прежние 20 баллов, без фиксированного требования «200 картинок».
Сначала согласуем задачу, классы и небольшой пилот. После пилота — разумный объём и разнообразие.
Запишите спорный случай и решение в инструкцию, сохраните метки и происхождение данных.
В следующем блоке добавим воспроизводимое групповое разбиение и правила аугментации.

**Результат этой пары:** учебные JSON/CSV в outputs и черновик инструкции своего проекта.
Не надо загружать в открытый репозиторий лица, номера, чужие изображения или служебные кадры только потому,
что их удобно размечать. Для каждого источника сначала проверьте условия и допустимый способ выдачи примера.
GitHub необязателен; папка или архив дают те же возможности сдачи.

**Резерв 10–15 минут.** Два человека независимо предлагают рамку чашки на этом фото, затем объясняют расхождение.
Не подглядывайте в чужую рамку до завершения. Исправьте одно предложение инструкции и повторите опыт.
Если работаете один — вернитесь к фото после другого задания и сравните два собственных решения.

Инструменты на будущее: [CVAT](https://docs.cvat.ai/docs/annotation/manual-annotation/),
[Label Studio](https://labelstud.io/guide/). Это варианты рабочего интерфейса, а не требование зарегистрироваться сегодня.
